[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/moeabench/blob/main/examples/example_18.ipynb)

# Example 18: Clinical Report & MaOP Calibration
----------------------------------------------

This tutorial demonstrates how to handle **Many-Objective Optimization Problems (MaOP)** where clinical baselines are not pre-calculated in the library registry.

We will cover:
1. Identifying the `MISSING_BASELINE` diagnostic status.
2. Generating a local baseline via `mb.calibrate()`.
3. Using sidecars for high-fidelity **Clinical Reports**.

In [ ]:
from moeabench import mb
import numpy as np

mb.system.version()

## 1. The MaOP Challenge

MoeaBench ships with authorized clinical baselines for standard 2D and 3D problems. However, for $M > 3$, the statistical noise floors are highly dependent on the problem topology and population size ($K$).

Let's look at a 4-objective DTLZ2 problem.

In [ ]:
mop = mb.mops.DTLZ2(M=4)
exp = mb.experiment(mop=mop, moea=mb.moeas.NSGA3(population=40, generations=20))
exp.run(repeat=1)

# Attempting an audit
audit_raw = mb.diagnostics.audit(exp)

print(f"Audit Status: {audit_raw.status.name}")
print(f"Description: {audit_raw.description}")

## 2. Local Calibration

To resolve the `MISSING_BASELINE` status, we use the `calibrate` utility. This establishes the physical resolution limits (the "Blur Radius") for this specific $M=4$ configuration.

In [ ]:
# Calibrate the problem locally
# This generates a sidecar JSON file with the noise floor distributions.
sidecar_path = "dtlz2_m4_demo.json"

mb.calibrate(
    mop, 
    size=40, 
    source_baseline=sidecar_path,
    force=True
)

print(f"\nSidecar generated at: {sidecar_path}")

## 3. Clinical Report with Sidecars

Once calibrated, we can use the `source_baseline` parameter to perform a high-fidelity Clinical Report.

In [ ]:
# Re-audit using the local sidecar
audit_fixed = mb.diagnostics.audit(exp, source_baseline=sidecar_path)

print("--- Calibrated Clinical Report ---")
audit_fixed.report()